# Importing Libraries

In [ ]:
import pandas as pd 
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt 
import seaborn as sns
from sklearn.utils import class_weight
from sklearn.preprocessing import FunctionTransformer  
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import RandomOverSampler
from sklearn.metrics import classification_report
from sklearn.decomposition import PCA
from tensorflow.keras.utils import to_categorical
from sklearn.neighbors import KNeighborsClassifier 
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from tensorflow.keras.optimizers import Adam
from sklearn.tree import DecisionTreeClassifier
import tensorflow
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow import keras
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense
from sklearn.metrics import accuracy_score

# Reading Data

In [ ]:
import pandas as pd 
import numpy as np
import os
cs = pd.read_csv("Credit Score Classification Dataset.csv")

In [ ]:
cs.head()

# Exploring Data

In [ ]:
cs.shape

In [ ]:
cs["Credit Score"].value_counts()

In [ ]:
sns.countplot(data=cs , x='Credit Score')
plt.title('Credit Scores')

In [ ]:
cs.describe()

# features name

In [ ]:
cs.columns

In [ ]:
cs["Credit Score"].value_counts()

# Missing Values

In [ ]:
print('Missing data sum :')
print(cs.isnull().sum())

print('\nMissing data percentage (%):')
print(cs.isnull().sum()/cs.count()*100)

# Seperate cat and numerical Features

In [ ]:
cat_features = [feature for feature in cs.columns if cs[feature].dtypes == 'O']
print('Number of cat variables: ', len(cat_features))
print('*'*80)
print('cat variables column name:',cat_features)

In [ ]:
numerical_features = [feature for feature in cs.columns if cs[feature].dtypes != 'O']
print('Number of numerical variables: ', len(numerical_features))
print('*'*80)
print('numerical Variables Column: ',numerical_features)

# Checking Duplicating Values

In [ ]:
cs.duplicated().sum()

In [ ]:
cs['Gender'].unique()

In [ ]:
cs['Education'].unique()

In [ ]:
cs['Marital Status'].unique()

In [ ]:
cs['Home Ownership'].unique()

In [ ]:
cs['Credit Score'].unique()

In [ ]:
cs_encoded = pd.get_dummies(cs, drop_first=True)
corr = cs_encoded.corr()
plt.figure(figsize=(10, 8))
sns.heatmap(data=corr, annot=True, cmap='Spectral').set(title="Correlation Matrix")

In [ ]:
corr_matrix = cs_encoded.corr().round(2)
corr_matrix              

In [ ]:
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
plt.figure(figsize=(10,10))
sns.heatmap(corr_matrix, center=0, vmin=-1, vmax=1, mask=mask, annot=True, cmap='coolwarm')
plt.show()

Visualizing cat Features
-

In [ ]:
for col in cat_features[:]:
    plt.figure(figsize=(6,3), dpi=100)
    sns.countplot(data=cs,x=col,hue ='Credit Score',palette='gist_rainbow_r')
    plt.legend(loc=(1.05,0.5))

# Barplot of numerical features:

-

In [ ]:
import warnings
warnings.filterwarnings('ignore')
for col in numerical_features:
    plt.figure(figsize=(6,3), dpi=100)
    sns.barplot(data=cs,x='Credit Score',y=col,palette='gist_rainbow_r')

Converting cat features into numerical
-

In [ ]:
train_data_cat = cs.select_dtypes("object")
train_data_num = cs.select_dtypes("number")

In [ ]:
train_data_cata_encoded = pd.get_dummies(train_data_cat, columns = train_data_cat.columns.to_list())
train_data_cata_encoded.head()

In [ ]:
data=pd.concat([train_data_cata_encoded,train_data_num],axis=1,join="outer")
data.head()

# seperate dependant and independant feature

In [ ]:
y = cs['Credit Score']
x = cs.drop('Credit Score', axis = 1)

In [ ]:
print(x.shape)
print(y.shape)

# scailing the data

In [ ]:
x = cs.select_dtypes(include=['number']) 
sc = StandardScaler()
x_scaled = sc.fit_transform(x)

In [ ]:
x

# Splitting data into Training and Testing

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, confusion_matrix,classification_report
from sklearn.svm import SVC
import pickle
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import GradientBoostingClassifier
from xgboost import XGBClassifier, plot_importance
from sklearn.model_selection import GridSearchCV, cross_val_score, StratifiedKFold, learning_curve

# Splitting the dataset

- training data 70%
- testing data 30%

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.3, random_state=7)
X_train.shape, X_test.shape

# Building Classifiers

In [ ]:
accuracy = {}

# Logistic Regression

In [ ]:
lr = LogisticRegression(max_iter=200)
lr.fit(X_train, y_train)
y_pred1 = lr.predict(X_test)
print(accuracy_score(y_test, y_pred1))
accuracy[str(lr)] = accuracy_score(y_test, y_pred1)*100

# Confusion Matrix 

In [ ]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test,y_pred1)
conf_matrix = pd.DataFrame(data = cm,columns = ['Predicted:0','Predicted:1','Predicted:2'],index=['Actual:0','Actual:1','Actual:2'])
plt.figure(figsize = (8,5))
sns.heatmap(conf_matrix, annot = True,fmt = 'd',cmap = "YlGnBu")

# Classification Report

In [ ]:
print(classification_report(y_test,y_pred1))

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(y_test,y_pred1, zero_division=1))  # sets precision/recall to 1 instead of 0


# Predicting

In [ ]:
y_pred_test = lr.predict(X_test)

test = pd.DataFrame({
    'Actual':y_test,
    'Y test predicted':y_pred_test
})

In [ ]:
test.sample(10)

# DecisionTreeClassifier

In [ ]:
dtc = DecisionTreeClassifier(max_depth=3)
dtc.fit(X_train, y_train)
y_pred2 = dtc.predict(X_test)
print(accuracy_score(y_test, y_pred2))
accuracy[str(dtc)] = accuracy_score(y_test, y_pred2)*100

In [ ]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test,y_pred2)
conf_matrix = pd.DataFrame(data = cm,columns = ['Predicted:0','Predicted:1','Predicted:2'],index=['Actual:0','Actual:1','Actual:2'])
plt.figure(figsize = (8,5))
sns.heatmap(conf_matrix, annot = True,fmt = 'd',cmap = "YlGnBu")

# Classification Report

In [ ]:
print(classification_report(y_test,y_pred2))

# Prediction

In [ ]:
y_pred_test = dtc.predict(X_test)

test = pd.DataFrame({
    'Actual':y_test,
    'Y test predicted':y_pred_test
})

In [ ]:
test.head(5)

# Random Forest Classifier

In [ ]:
rfc = RandomForestClassifier(max_depth=5)
rfc.fit(X_train, y_train)
y_pred3 = rfc.predict(X_test)
print(accuracy_score(y_test, y_pred3))
accuracy[str(rfc)] = accuracy_score(y_test, y_pred3)*100

In [ ]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test,y_pred3)
conf_matrix = pd.DataFrame(data = cm,columns = ['Predicted:0','Predicted:1','Predicted:2'],index=['Actual:0','Actual:1','Actual:2'])
plt.figure(figsize = (8,5))
sns.heatmap(conf_matrix, annot = True,fmt = 'd',cmap = "YlGnBu")

# Gradient Boosting Classifier

In [ ]:
gbc = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1)
gbc.fit(X_train, y_train)
y_pred4 = gbc.predict(X_test)
print(accuracy_score(y_test, y_pred4))
accuracy[str(gbc)] = accuracy_score(y_test, y_pred4)*100

In [ ]:
from sklearn.metrics import confusion_matrix
cm=confusion_matrix(y_test,y_pred4)
conf_matrix=pd.DataFrame(data=cm,columns=['Predicted:0','Predicted:1','Predicted:2'],index=['Actual:0','Actual:1','Actual:2'])
plt.figure(figsize = (8,5))
sns.heatmap(conf_matrix, annot=True,fmt='d',cmap="YlGnBu")

# SVM 

In [ ]:
svc = SVC()
svc.fit(X_train, y_train)
y_pred5 = svc.predict(X_test)
print(accuracy_score(y_test, y_pred5))
accuracy[str(svc)] = accuracy_score(y_test, y_pred5)*100

In [ ]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test,y_pred5)
conf_matrix = pd.DataFrame(data = cm,columns = ['Predicted:0','Predicted:1','Predicted:2'],index=['Actual:0','Actual:1','Actual:1'])
plt.figure(figsize = (8,5))
sns.heatmap(conf_matrix, annot = True,fmt = 'd',cmap = "YlGnBu")

# Accuracy

In [ ]:
accuracy

# Handling this data using SMOTE

In [ ]:
from imblearn.over_sampling import SMOTE

In [ ]:
smote = SMOTE()
x1, y1 = smote.fit_resample(x, y)
x1.shape, y1.shape 

In [ ]:
df=pd.DataFrame(x1)
df.head()

# Splitting the oversampling data

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(x1,y1, test_size=0.3 ,shuffle = True,random_state = 3)

In [ ]:
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

# Logistic Regression

In [ ]:
lr = LogisticRegression(max_iter=200)
lr.fit(X_train, y_train)
y_pred1 = lr.predict(X_test)
print(accuracy_score(y_test, y_pred1))
accuracy[str(lr)] = accuracy_score(y_test, y_pred1)*100

In [ ]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test,y_pred1)
conf_matrix = pd.DataFrame(data = cm,columns = ['Predicted:0','Predicted:1','Predicted:2'],index=['Actual:0','Actual:1','Actual:2'])
plt.figure(figsize = (8,5))
sns.heatmap(conf_matrix, annot = True,fmt = 'd',cmap = "YlGnBu")

# Classification Report

In [ ]:
print(classification_report(y_test,y_pred1))

# Prediction

In [ ]:
y_pred_test = lr.predict(X_test)

test = pd.DataFrame({
    'Actual':y_test,
    'Y test predicted':y_pred_test
})

In [ ]:
test.head()

# K-Nearest Neighbor Classifier

In [ ]:
knn_model = KNeighborsClassifier(n_neighbors=3)
knn_model.fit(X_train,y_train)
knn_predict = knn_model.predict(X_test)
print(accuracy_score(y_test, knn_predict))
accuracy[str(lr)] = accuracy_score(y_test, knn_predict)*100

In [ ]:
from sklearn.metrics import confusion_matrix
cm=confusion_matrix(y_test,knn_predict)
conf_matrix=pd.DataFrame(data=cm,columns=['Predicted:0','Predicted:1','Predicted:2'],index=['Actual:0','Actual:1','Acttual:2'])
plt.figure(figsize = (8,5))
sns.heatmap(conf_matrix, annot=True,fmt='d',cmap="YlGnBu")

# Classification Report 

In [ ]:
print(classification_report(y_test,knn_predict))

# Prediction

In [ ]:
y_pred_test = knn_model.predict(X_test)

test = pd.DataFrame({
    'Actual':y_test,
    'Y test predicted':y_pred_test
})

In [ ]:
test.sample(10)

# ANN

- Creating sequnetial ANN Network
- Creating 5 layers Network
- Activation is "Relu"
- Last layer is output layer
- Problem is binary classification thats way output node is 1 and activation is "sigmoid"

In [ ]:
df = pd.read_csv("Credit Score Classification Dataset.csv")  # Replace with your actual filename
df = df.dropna()

X = df.drop("Credit Score", axis=1)
X = pd.get_dummies(X)
y = df["Credit Score"]
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)      
y_categorical = to_categorical(y_encoded)       

X_train, X_test, y_train, y_test = train_test_split(X, y_categorical, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

model = Sequential()
model.add(Dense(64, activation='relu', input_shape=(X_train.shape[1],)))
model.add(Dense(32, activation='relu'))
model.add(Dense(3, activation='softmax'))  

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

early_stop = EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True)

history = model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)
loss, accuracy = model.evaluate(X_test, y_test)
print("Test Accuracy:", accuracy)

# Testing the Model

In [ ]:
model.evaluate(X_test, y_test)

In [ ]:
y_pred = model.predict(X_test)  

In [ ]:
y_pred = np.argmax(y_pred, axis=1)

In [ ]:
y_pred = np.round(y_pred).astype(int)

In [ ]:
print("y_test shape:", y_test.shape)
print("y_pred shape:", y_pred.shape)

In [ ]:
y_pred_probs = model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)
if len(y_test.shape) > 1 and y_test.shape[1] > 1:
    y_test_labels = np.argmax(y_test, axis=1)
else:
    y_test_labels = y_test
print(classification_report(y_test_labels, y_pred))

In [ ]:
model.fit(X_train, y_train, epochs=10, batch_size=32, validation_split=0.2)

# Confusion Matrix

In [ ]:
y_pred_labels = y_pred
if len(y_test.shape) > 1 and y_test.shape[1] > 1:
    y_test_labels = np.argmax(y_test, axis=1)
else:
    y_test_labels = y_test
cm = confusion_matrix(y_test_labels, y_pred_labels)
conf_matrix = pd.DataFrame(
    data=cm,
    columns=['Predicted: Low', 'Predicted: Medium', 'Predicted: High'],
    index=['Actual: Low', 'Actual: Medium', 'Actual: High']
)
plt.figure(figsize=(8, 5))
sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues")
plt.title("Confusion Matrix")
plt.ylabel("Actual")
plt.xlabel("Predicted")
plt.show()